# A2 — initial code checks

TrashCan-Material: U-Net scratch, SegFormer-B0 color off/on, seed 69420; split seed 36.
This notebook calls the shared repository modules. Default Run all performs **synthetic CPU checks only**.
No main training or official-test evaluation is enabled here. Draft optimization/budget awaits approval and pilot.

Use an existing checkout locally. In a fresh Colab runtime, upload the supplied `a2_initial_source.zip`
when the setup cell asks; the branch is local until the user publishes it. Raw data/weights are not in that ZIP.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys
import zipfile

IN_COLAB = 'google.colab' in sys.modules
if not IN_COLAB:
    try:
        import google.colab
        IN_COLAB = True
    except ImportError:
        pass
SOURCE_DIR = Path('/content/dlbench-a2') if IN_COLAB else Path.cwd()
if not IN_COLAB and not (SOURCE_DIR / 'src/dlbench/a2').exists():
    SOURCE_DIR = SOURCE_DIR.parent  # notebook opened in notebooks/
if IN_COLAB and not (SOURCE_DIR / 'src/dlbench/a2').exists():
    from google.colab import files
    uploaded = files.upload()  # choose a2_initial_source.zip
    matches = [Path(name) for name in uploaded if name.endswith('.zip')]
    if len(matches) != 1:
        raise ValueError('Upload exactly one A2 source ZIP')
    SOURCE_DIR.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(matches[0]) as archive:
        for name in archive.namelist():
            if not (SOURCE_DIR / name).resolve().is_relative_to(SOURCE_DIR.resolve()):
                raise ValueError('Unsafe archive path')
        archive.extractall(SOURCE_DIR)
assert (SOURCE_DIR / 'src/dlbench/a2').is_dir(), 'Point SOURCE_DIR to the A2 checkout'
if IN_COLAB:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-e', str(SOURCE_DIR) + '[a2]'])
os.chdir(SOURCE_DIR)
sys.path.insert(0, str(SOURCE_DIR / 'src'))
os.environ['PYTHONPATH'] = str(SOURCE_DIR / 'src')
import torch
torch.set_num_threads(2)

def cli(*args):
    subprocess.check_call([sys.executable, '-m', 'dlbench.a2.cli', *map(str, args)])

print('Source:', SOURCE_DIR, '| Torch:', torch.__version__)


## 1. Validate draft configs and synthetic contracts

SegFormer synthetic checks use the actual B0 architecture with **random** weights; no network download.
Set `CHECK_PRETRAINED = True` only to exercise the pinned ADE20K initialization explicitly.


In [ ]:
for name in ['unet', 'segformer_off', 'segformer_on']:
    cli('validate-config', '--config', f'configs/a2/{name}.py')
from dlbench.a2.cli import synthetic_smoke
from dlbench.a2.config import load_config
for name in ['unet', 'segformer_off']:
    print(synthetic_smoke(load_config(f'configs/a2/{name}.py')))
CHECK_PRETRAINED = False
if CHECK_PRETRAINED:
    print(synthetic_smoke(load_config('configs/a2/segformer_off.py'), pretrained=True))


## 2. Data paths and sampled native-mask checks

Mount Drive explicitly if needed. Set `CHECK_REAL_DATA = True` after setting the paths below.
The converter verifies annotation/category identity, video split and each loaded native target hash.
The bundled manifest is copied unchanged from the 5 October EDA; frames/COCO JSON remain external.


In [ ]:
CHECK_REAL_DATA = False
DATA_ROOT = Path('/content/drive/MyDrive/TrashCanMaterial')
MANIFEST = SOURCE_DIR / 'manifests/trashcan_material_seed36_v1.json'
# Local external EDA manifest also works: set MANIFEST to its actual path.
if MANIFEST.is_file():
    cli('check-manifest', '--manifest', MANIFEST)
if CHECK_REAL_DATA:
    for split in ['train', 'validation']:
        cli('check-data', '--config', 'configs/a2/unet.py', '--data-root', DATA_ROOT,
            '--manifest', MANIFEST, '--split', split, '--limit', 4)


## 3. Inspect an image-mask overlay

Enable after the sampled data check. This uses the same Dataset/preprocessing as training.
Ignored pixels are shown in gray. This is visual QA, not a prediction or training result.


In [ ]:
if CHECK_REAL_DATA:
    import matplotlib.pyplot as plt
    import numpy as np
    from dlbench.a2.data import TrashCanDataset
    cfg = load_config('configs/a2/unet.py')
    ds = TrashCanDataset(DATA_ROOT, MANIFEST, 'validation', cfg['preprocessing'])
    sample = ds[0]
    rgb = sample['image'].permute(1, 2, 0).numpy()
    rgb = np.clip(rgb * np.array(cfg['preprocessing']['std']) + np.array(cfg['preprocessing']['mean']), 0, 1)
    mask = sample['mask'].numpy()
    colors = plt.get_cmap('tab20')(np.minimum(mask, 16) / 19)[..., :3]
    colors[mask == 255] = 0.5
    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    for ax, content, title in zip(axes, [rgb, colors, .6 * rgb + .4 * colors], ['Image', 'Target', 'Overlay']):
        ax.imshow(content); ax.set_title(title); ax.axis('off')
    fig.suptitle(f"Image {sample['image_id']} / {sample['video_id']}")
    plt.show()


## 4. Optional tiny development integration

Keep disabled for initial-code review. When real data is ready, this trains on eight train/validation
images for **one development epoch** and labels artifacts accordingly. Use a new output folder for each run.
For SegFormer this path loads the pinned pretrained checkpoint. The same engine handles both models.
Main training remains gated by a frozen protocol plus approval/pilot evidence; this notebook has no main-run cell.


In [ ]:
RUN_DEVELOPMENT = False
DEV_CONFIG = 'configs/a2/unet.py'
DEV_OUTPUT = Path('/content/drive/MyDrive/dlbench_a2/development/U-dev-001')
if RUN_DEVELOPMENT:
    assert CHECK_REAL_DATA, 'Complete and inspect real-data checks first'
    cli('train', '--config', DEV_CONFIG, '--data-root', DATA_ROOT, '--manifest', MANIFEST,
        '--output', DEV_OUTPUT, '--development', '--limit', 8, '--epochs', 1,
        '--device', 'cuda' if torch.cuda.is_available() else 'cpu')


## Readiness after initial code

Review real overlays, integrate a real forward/backward batch, verify tiny-subset overfit, obtain proposal
approval evidence, run a GPU capacity pilot, and freeze optimizer/loss/budget before main U/S0/S1 runs.
S0/S1 must have identical initial-state hashes and paired settings. Decisions use validation, never test.
See `docs/a2_initial_code.md` for resume/evaluation commands and precise metric conventions.
